# Implement AdamW from Scratch - SOLUTION

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google

---

### Problem Statement

AdamW (**Loshchilov & Hutter, 2017**) fixes a flaw in the usual way weight decay is bolted onto Adam. With **L2 regularization** the penalty is added to the *gradient* (`g ← g + w·θ`), so it passes through the adaptive denominator — weights with large gradients get regularized *less*. **AdamW decouples** the decay from the gradient entirely:

```
θ ← θ − lr·w·θ        (directly on the parameter)
```

It is the default optimizer for training transformers.

### Tasks

1. `MyAdamW` — Adam's update plus weight decay applied **directly to the parameters**, not through the gradient, following PyTorch's `torch.optim.Optimizer` interface.

### References

- AdamW paper: https://arxiv.org/abs/1711.05101

In [ ]:
import math
import torch
from torch.optim import Optimizer


## Part 2: AdamW

AdamW fixes a flaw in the usual way weight decay is bolted onto Adam. With **L2 regularization** the penalty is added to the *gradient* (`g ← g + w·θ`), so it passes through the adaptive denominator — weights with large gradients get regularized *less*. **AdamW decouples** the decay from the gradient entirely:

```
θ ← θ − lr·w·θ                     (directly on the parameter)
θ ← θ − lr·m̂_t / (√v̂_t + ε)        (the usual Adam step)
```

Note the decay is multiplied by `lr`, so the effective decay rate is `lr * weight_decay` — and a parameter with a **zero gradient must still decay**.


In [ ]:
class MyAdamW(Optimizer):
    """
    AdamW optimizer — Adam with decoupled weight decay.

    Args:
        params:       iterable of parameters to optimize
        lr:           learning rate (default 1e-3)
        betas:        coefficients for the moment estimates (default (0.9, 0.999))
        eps:          term for numerical stability (default 1e-8)
        weight_decay: weight decay coefficient (default 1e-2)
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2):
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if weight_decay < 0.0:
            raise ValueError(f"Invalid weight_decay: {weight_decay}")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad

                if grad.is_sparse:
                    raise RuntimeError("MyAdamW does not support sparse gradients")

                state = self.state[p]

                # State initialization (same as Adam)
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)

                exp_avg: torch.Tensor = state['exp_avg']
                exp_avg_sq: torch.Tensor = state['exp_avg_sq']
                state['step'] += 1

                # Biased moment estimates and bias corrections
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)
                step_size = lr / bias_correction1

                # Decoupled weight decay — directly on the parameter,
                # never through the gradient
                p.mul_(1 - lr * weight_decay)

                # Adam update
                p.addcdiv_(exp_avg, denom, value=-step_size)

        return loss


## Validation

`MyAdamW` is compared step-for-step against `torch.optim.AdamW` on a quadratic bowl, and decoupled weight decay is verified to *differ* from Adam + L2 regularization.

Until your implementation is in place these tests will fail — that is expected.


In [ ]:
def test_adamw():
    """Compare MyAdamW with torch.optim.AdamW. Key: AdamW != Adam with wd."""
    print("Testing AdamW...", end=" ")

    torch.manual_seed(42)
    D = 64
    target = torch.linspace(-1, 1, D)

    x_ref = torch.zeros(D, requires_grad=True)
    opt_ref = torch.optim.AdamW([x_ref], lr=0.1, betas=(0.9, 0.999),
                                eps=1e-8, weight_decay=0.01)

    x_my = torch.zeros(D, requires_grad=True)
    opt_my = MyAdamW([x_my], lr=0.1, betas=(0.9, 0.999),
                     eps=1e-8, weight_decay=0.01)

    for _ in range(100):
        loss_ref = ((x_ref - target) ** 2).mean()
        opt_ref.zero_grad()
        loss_ref.backward()
        opt_ref.step()

        loss_my = ((x_my - target) ** 2).mean()
        opt_my.zero_grad()
        loss_my.backward()
        opt_my.step()

    diff = (x_ref - x_my).abs().max().item()
    if diff < 1e-5:
        print(f"PASS  (max parameter diff: {diff:.2e})")
    else:
        print(f"FAIL  (max parameter diff: {diff:.2e})")




In [ ]:
def test_adamw_weight_decay():
    """AdamW's decoupled weight decay differs from Adam + L2 regularization."""
    print("Testing AdamW decoupled weight decay...", end=" ")

    torch.manual_seed(42)
    D = 16
    target = torch.randn(D)
    wd = 0.1

    x_adamw = torch.zeros(D, requires_grad=True)
    opt_adamw = torch.optim.AdamW([x_adamw], lr=0.1, weight_decay=wd)

    x_adam_l2 = torch.zeros(D, requires_grad=True)
    opt_adam_l2 = torch.optim.Adam([x_adam_l2], lr=0.1)

    for _ in range(50):
        loss_adamw = ((x_adamw - target) ** 2).mean()
        opt_adamw.zero_grad()
        loss_adamw.backward()
        opt_adamw.step()

        # Adam + L2: weight decay added to the gradient (NOT the same as AdamW)
        loss_l2 = ((x_adam_l2 - target) ** 2).mean()
        opt_adam_l2.zero_grad()
        loss_l2.backward()
        with torch.no_grad():
            x_adam_l2.grad.add_(wd * x_adam_l2)
        opt_adam_l2.step()

    diff = (x_adamw - x_adam_l2).abs().max().item()
    if diff > 1e-6:
        print(f"PASS  (AdamW != Adam+L2, max diff: {diff:.4f})")
    else:
        print("FAIL  (AdamW and Adam+L2 gave identical results)")


